<a href="https://colab.research.google.com/github/Sundeep-Parmar-UC/AgeCare-ACIL/blob/main/ACIL_Excel_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, we need to install the `openpyxl` library, which pandas uses to read and write Excel files.

In [197]:
import pandas as pd
import re # Import the regular expression module
import colorsys
import numpy as np

!pip install openpyxl
!pip install xlsxwriter


In [198]:
#Global Variable Definitions

#Loaded worksheets: A dictionary where keys are source filenames and values are dictionaries of DataFrames (each representing a sheet from that file).
all_worksheets_df = {} # Re-initialize to ensure it's empty before populating for multiple files

#Source File Name:
SourceFile = []
SourceFile.append("ACIL-source.xlsx")
SourceFilePath = "/content/"

#Destination File Name:
DestinationFile = "Carlos-Summary.xlsx"
DestinationFilePath = "/content/"

new_excel_file_path = DestinationFilePath + DestinationFile

#set Month to summarize
MonthList = {}
RevisedMonthList = {}

#LedgerColumn,  the column that contains the financial categories
LedgerColumn = -1

#MonthColumn,  the column that contains the finanacial actual data for the month
MonthColumn = -1

#SiteList
SiteList = {}
#SiteList = ["BBC", "BJP", "BMT", "BMC", "BMR", "BRV", "BST", "BGW", "BHR", "BJB", "BLV", "BRC", "BSD", "BCT", "BMS"]

#Financial Categories (Actuals)
FinCatList = ["Sick Time", "Overtime", "Purchased Hours", "- Added Care", "Purchased Hours total", "Repair", "Maintenance", "Other (RnM)", "6500:Care Supplies", "6600:Clinical supplies", "Nursing Supplies - Added Care Expense", "SUPPLIES", "Added-Care", "6700:Repair & Maintenance", "SUM"]

#Financial Categories (Budget)
FinCatListBudget = ["Sick Time","Overtime","Purchased Hours","Repair","Maintenance","6500:Care Supplies","6600:Clinical supplies","SUPPLIES","6700:Repair & Maintenance","SUM"]

#Variance Categories
VarCatList = ["Sick Time", "Overtime", "Purchased Hours", "Repair", "Maintenance", "6500:Care Supplies", "6600:Clinical supplies", "SUPPLIES"]

#defined month sequence for improvement colours
MonthSequence = ["Jan 2025", "Feb 2025", "Mar 2025", "Apr 2025", "May 2025", "Jun 2025", "Jul 2025", "Aug 2025", "Sep 2025", "Oct 2025", "Nov 2025", "Dec 2025", "Jan 2026", "Feb 2026", "Mar 2026", "Apr 2026", "May 2026", "Jun 2026", "Jul 2026", "Aug 2026", "Sep 2026", "Oct 2026", "Nov 2026", "Dec 2026","Jan 2027", "Feb 2027", "Mar 2027", "Apr 2027", "May 2027", "Jun 2027", "Jul 2027", "Aug 2027", "Sep 2027", "Oct 2027", "Nov 2027", "Dec 2027","Jan 2028", "Feb 2028", "Mar 2028", "Apr 2028", "May 2028", "Jun 2028", "Jul 2028", "Aug 2028", "Sep 2028", "Oct 2028", "Nov 2028", "Dec 2028","Jan 2029", "Feb 2029", "Mar 2029", "Apr 2029", "May 2029", "Jun 2029", "Jul 2029", "Aug 2029", "Sep 2029", "Oct 2029", "Nov 2029", "Dec 2029", "Jan 2030", "Feb 2030", "Mar 2030", "Apr 2030", "May 2030", "Jun 2030", "Jul 2030", "Aug 2030", "Sep 2030", "Oct 2030", "Nov 2030", "Dec 2030","Jan 2031", "Feb 2031", "Mar 2031", "Apr 2031", "May 2031", "Jun 2031", "Jul 2031", "Aug 2031", "Sep 2031", "Oct 2031", "Nov 2031", "Dec 2031","Jan 2032", "Feb 2032", "Mar 2032", "Apr 2032", "May 2032", "Jun 2032", "Jul 2032", "Aug 2032", "Sep 2032", "Oct 2032", "Nov 2032", "Dec 2032"]

#defined site sequence for final columns order
SiteSequence = ['CL','GL','ACILDS','MN','SN','OM','VV','HC','SW','ST','SP','SG','WH','BCM','CPL','MOM','MIM','SR','BBC','BJP','BMT','BMC','BMR','BRV','BST','BGW','BHR','BJB','BLV','BRC','BSD','BCT','BMS','MAC','IAM','IAR','IEM','ILD','IPG','IPH','IRO','ITL','RBR','RGO','RSM','RWB','RWG','RWH','RWL','RWW','MGS','MBC','MQG','MRG','MVF']

#Flag for ACIL File
ACILFlag = {}
LedgerColumnsACIL = {}
MonthColumnsActualsACIL = {}
MonthColumnsBudgetsACIL = {}
MonthsListPossible = ['Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar']


Now, let's load an Excel file into a pandas DataFrame. You can specify a particular worksheet by name or by index (0-based) using the `sheet_name` parameter in `pd.read_excel()`.

In [199]:
unique_site_names = set()
unique_month_names = set()

for filename in SourceFile: # Iterate through each file in SourceFile list
    full_file_path = SourceFilePath + filename

    # Read all sheets from the current Excel file
    # sheet_name=None returns a dictionary where keys are sheet names and values are DataFrames
    # header=None tells pandas that the file does not have a header row, so row 0 is data.
    current_file_worksheets = pd.read_excel(full_file_path, sheet_name=None, header=None)
    all_worksheets_df[filename] = current_file_worksheets # Store under the filename as per new definition

    # Populate SiteList and MonthList from sheets of the current file
    for sheet_name in current_file_worksheets.keys():
        #print("sheet_name:",sheet_name)
        if (re.fullmatch(r'^[A-Z]{2,3}$', sheet_name) and sheet_name != "YTD"): # Check for exactly two or three capital letters
            unique_site_names.add(sheet_name)
        elif (sheet_name == "ACILDS"): # Check for MMM 20XX Budget pattern
            unique_site_names.add(sheet_name)

        # Check for month budget worksheets and extract the month
        if re.fullmatch(r'^[A-Z][a-z]{2} 20[0-9]{2} Budget$', sheet_name): # Check for MMM 20XX Budget pattern
            unique_month_names.add(sheet_name[:8])

    # Convert sets to sorted lists for consistent output
    # Sort unique_site_names based on the order in SiteSequence
    SiteList[filename] = sorted(
        [site for site in unique_site_names if site in SiteSequence],
        key=lambda site: SiteSequence.index(site)
    )
    # Sort unique_month_names based on the order in MonthSequence
    MonthList[filename] = sorted(
        [month for month in unique_month_names if month in MonthSequence],
        key=lambda month: MonthSequence.index(month)
    )

    print(f"Successfully loaded all worksheets from file: {filename}.")
    print(f"Worksheets identified as Site Locations (two or three capital letters): {SiteList[filename]}")
    print(f"Months identified from 'MMM 20XX Budget' worksheets: {MonthList[filename]}")

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Successfully loaded all worksheets from file: ACIL-source.xlsx.
Worksheets identified as Site Locations (two or three capital letters): ['CL', 'GL', 'ACILDS', 'MN', 'SN', 'OM', 'VV', 'HC']
Months identified from 'MMM 20XX Budget' worksheets: []


In [200]:
# Look through entire DataWorkSheet and find column of value "Ledger Account"
def FindLedgerColumn(DataWorkSheet):

  ledger_account_col = 1000

  for r_idx, row in DataWorkSheet.iterrows():
      for c_idx, value in row.items():
          if isinstance(value, str) and value == "Ledger Account":
              if(c_idx < ledger_account_col):
                ledger_account_col = c_idx
              continue

  return ledger_account_col


In [201]:
# Look through entire DataWorkSheet and find column of Month actuals
def FindMonthActualsColumn(DataWorkSheet,SummaryMonth):

  month_row = -1
  month_col = -1

  for r_idx, row in DataWorkSheet.iterrows():
      for c_idx, value in row.items():
          # Make comparison robust by stripping whitespace and converting to uppercase
          if isinstance(value, str) and value.strip().upper() == SummaryMonth.strip().upper():
              month_row = r_idx
              month_col = c_idx
              break # Exit inner loop once found
      if month_row != -1:
          break # Exit outer loop once found

  return month_row, month_col # Return both row and column

In [202]:
#set Year string variable
YearString = ""
YearsOptions = ['2024','2025','2026','2027','2028','2029','2030','2031','2032']

#checking for ACIL file
for filename in SourceFile:

    # Check if filename exists in MonthList and if its corresponding list is not empty
    if filename in MonthList and MonthList[filename]:
        ACILFlag[filename] = False # It has specific budget months, not a generic ACIL file
        continue

    else:
        ACILFlag[filename] = True # Either filename not in MonthList or MonthList[filename] is empty, treat as generic ACIL

        # Initialize dictionaries for the current filename if they don't exist
        if filename not in LedgerColumnsACIL:
            LedgerColumnsACIL[filename] = {}
        if filename not in MonthColumnsActualsACIL:
            MonthColumnsActualsACIL[filename] = {}
        if filename not in MonthColumnsBudgetsACIL:
            MonthColumnsBudgetsACIL[filename] = {}

        #open each one identified site
        for site in SiteList[filename]:
            # Initialize site-specific dictionaries for month-based columns
            if site not in MonthColumnsActualsACIL[filename]:
                MonthColumnsActualsACIL[filename][site] = {}
            if site not in MonthColumnsBudgetsACIL[filename]:
                MonthColumnsBudgetsACIL[filename][site] = {}

            # Get the DataFrame for the current site
            ACIL_Site_df = all_worksheets_df[filename][site]

            #if Year string is blank then search current site worksheet for fiscal year
            if YearString == "":
              for c_idx in range(0, 4):
                for r_idx in range(0,13):
                  cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                  if isinstance(cell_value, str):
                      StartingFour = cell_value[0:4]
                      if StartingFour in YearsOptions:
                        YearString = StartingFour
                        break
                  if YearString != "":
                    break
                if YearString != "":
                  break

            # Find LedgerColumn
            current_ledger_col = FindLedgerColumn(ACIL_Site_df)
            LedgerColumnsACIL[filename][site] = current_ledger_col

            #Find Months and then Actual and Budget columns
            for monthcheck in MonthsListPossible:
                month_header_row, current_month_col = FindMonthActualsColumn(ACIL_Site_df, monthcheck) # Get both row and col
                if(current_month_col != -1): # If a column for the month was found
                    actuals_found = False
                    budget_found = False
                    actuals_value_found = False
                    budget_value_found = False
                    actuals_column_candidate = -1
                    budget_column_candidate = -1


                    # Look down the current_month_col for "Actuals" and "Budget" labels
                    # Start searching from the row where the month header was found, or the row after it.
                    for c_idx in range(current_month_col, current_month_col+2):

                      for r_idx in range(month_header_row, month_header_row+5):
                        cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                        if isinstance(cell_value, str):
                            upper_value = cell_value.strip().upper()
                            if upper_value == "ACTUALS" and actuals_found is False:
                                actuals_column_candidate = c_idx
                                actuals_found = True

                            if upper_value == "BUDGET" and budget_found is False:
                                budget_column_candidate = c_idx
                                budget_found = True

                        if(actuals_found and actuals_column_candidate == c_idx):
                            #check if current cell contains financial data
                            cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                            if isinstance(cell_value, (int, float)) and pd.notna(cell_value):
                                actuals_value_found = True

                        if(budget_found and budget_column_candidate == c_idx):
                            #check if current cell contains financial data
                            cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                            if isinstance(cell_value, (int, float)) and pd.notna(cell_value):
                                budget_value_found = True

                        if(actuals_found and budget_found and actuals_value_found and budget_value_found):
                            MonthColumnsActualsACIL[filename][site][monthcheck]= actuals_column_candidate
                            MonthColumnsBudgetsACIL[filename][site][monthcheck] = budget_column_candidate
                            AddThisMonth = monthcheck + " " + YearString
                            if AddThisMonth not in MonthList[filename]:
                              MonthList[filename].append(AddThisMonth)
                              if monthcheck == "Dec":
                                #add 1 to Yearstring
                                YearString = str(int(YearString) + 1)


To view the original formulas in the Excel cells, we'll use `openpyxl` directly. This will allow us to access the `formula` attribute of each cell.

In [203]:
#given 1)SiteWorksheet 2)Ledger Column 3)Month Column 4)Category
# calculate sum of all values in Month Column, when Category appears in Ledge column from SiteWorksheet

def calculate_sum_by_category(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category):
    """
    Calculates the sum of values in a Month Column for a given Category in a Ledger Column.

    Args:
        DataWorkSheet (pd.DataFrame): The DataFrame representing the site worksheet.
        LedgerColumnIndex (int): The index of the column containing ledger categories.
        MonthColumnIndex (int): The index of the column containing monthly financial data.
        Category (str): The category to filter by in the Ledger Column.

    Returns:
        float: The sum of values in the Month Column for the specified Category.
    """
    # Filter the DataFrame for rows where the LedgerColumnIndex matches the Category
    # Ensure the column exists and handle potential non-string types gracefully
    filtered_df = DataWorkSheet[
        (DataWorkSheet[LedgerColumnIndex].astype(str) == Category)
    ]

    # Sum the values in the MonthColumnIndex from the filtered DataFrame
    # Convert to numeric, coercing errors to NaN, then drop NaNs before summing
    total_sum = filtered_df[MonthColumnIndex].apply(pd.to_numeric, errors='coerce').sum()

    return total_sum



In [204]:
#given 1)SiteWorksheet 2)Ledger Column 3)Month Column 4)Category
#determine which function to use to generate value
#then return value from that function

def calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category):

    StraightSumCategory = ["Sick Time", "Overtime", "Purchased Hours", "6500:Care Supplies", "6600:Clinical supplies", "Nursing Supplies - Added Care Expense", "Added-Care", "6700:Repair & Maintenance"]
    if Category in StraightSumCategory:
        return calculate_sum_by_category(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category)

    elif Category == "Purchased Hours total":
        return (calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "Purchased Hours") - calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "- Added Care"))

    elif Category == "SUPPLIES":
        return (calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "6500:Care Supplies") + calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "6600:Clinical supplies") - calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "Nursing Supplies - Added Care Expense"))

    elif Category == "SUM":
        return (calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "Repair") + calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "Maintenance") + calculate_value(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, "Other (RnM)"))

    elif Category == "- Added Care":
        return calculate_sum_of_Added_Care(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex)

    elif Category == "Repair" or Category == "Maintenance":
        return calculate_sum_of_Repair_Maintence(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category)

    elif Category == "Other (RnM)":
        return calculate_sum_of_Other_RnM(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category)

    else:
        return -1000000




In [205]:
def calculate_sum_of_Added_Care(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex):

    # Corrected logic for '- Added Care' directly within this cell
    total_added_care_sum = 0
    # Assuming the maximum number of rows to check is reasonable, e.g., DataWorkSheet.shape[0]
    for i in range(DataWorkSheet.shape[0] - 1): # Iterate up to second to last row
        try:
            # Check if current row matches the Category:"Added Care HCA"
            current_row_val = str(DataWorkSheet.iloc[i, LedgerColumnIndex]) # Convert to string for comparison

            # Check if the NEXT row matches "Purchased Hours"
            next_row_val = str(DataWorkSheet.iloc[i + 1, LedgerColumnIndex]) # Convert to string for comparison

            if current_row_val == "Added Care HCA" and next_row_val == "Purchased Hours":
                # Add the NEXT row's value from the Month column
                # Convert to float to handle numeric addition safely
                total_added_care_sum += pd.to_numeric(DataWorkSheet.iloc[i + 1, MonthColumnIndex], errors='coerce')
        except IndexError:
            # Handle cases where i+1 goes out of bounds defensively, though shape[0]-1 handles most
            continue
        except ValueError:
            # Skip if conversion to numeric fails
            continue

    return total_added_care_sum

In [206]:
def calculate_sum_of_Repair_Maintence(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category):
    total_sum = 0
    in_target_section = False

    for i in range(DataWorkSheet.shape[0]): # Iterate over all rows of the DataFrame
        try:
            # Safely extract and clean text from LedgerColumn using .iloc
            cell_a_value = str(DataWorkSheet.iloc[i, LedgerColumnIndex]).strip()

            # 1. Update the state machine (SCAN logic)
            if cell_a_value == "6700:Repair & Maintenance":
                in_target_section = True
            elif in_target_section and ":" in cell_a_value:
                # Found a colon in a new section header; turn off the switch
                in_target_section = False

            # 2. Check if the row meets both criteria to be summed
            starts_with_repair = cell_a_value.startswith(Category)

            if in_target_section and starts_with_repair:
                # Extract value from MonthColumn and add to total, handling non-numeric data
                total_sum += pd.to_numeric(DataWorkSheet.iloc[i, MonthColumnIndex], errors='coerce')

        except (IndexError, ValueError, TypeError):
            # Skips blank cells, formatting issues, or short sheets safely
            continue

    return total_sum

In [207]:
def calculate_sum_of_Other_RnM(DataWorkSheet, LedgerColumnIndex, MonthColumnIndex, Category):
    total_sum = 0
    in_target_section = False

    for i in range(DataWorkSheet.shape[0]): # Iterate over all rows of the DataFrame
        try:
            # Safely extract and clean text from LedgerColumn using .iloc
            cell_a_value = str(DataWorkSheet.iloc[i, LedgerColumnIndex]).strip()

            # 1. Update the state machine (SCAN logic)
            if cell_a_value == "6700:Repair & Maintenance":
                in_target_section = True
                continue
            elif in_target_section and ":" in cell_a_value:
                # Found a colon in a new section header; turn off the switch
                in_target_section = False
                continue
            elif in_target_section and "Marketing & Advertising" in cell_a_value:
                # Found a title of next section; turn off the switch
                in_target_section = False
                continue


            # 2. Check if the row meets both criteria to be summed
            # The category should not start with "Repair" or "Maintenance"
            is_other_rnm_category = not (cell_a_value.startswith("Repair") or cell_a_value.startswith("Maintenance"))

            if in_target_section and is_other_rnm_category:
                # Extract value from MonthColumn and add to total, handling non-numeric data
                total_sum += pd.to_numeric(DataWorkSheet.iloc[i, MonthColumnIndex], errors='coerce')
                print("total_sum:",total_sum)
                print("DataWorkSheet.iloc[i, MonthColumnIndex], i:",i,"  MonthColumnIndex: ",MonthColumnIndex,"  cell_a_value:",cell_a_value)
        except (IndexError, ValueError, TypeError):
            # Skips blank cells, formatting issues, or short sheets safely
            continue

    return total_sum

In [208]:
# This will store all row data for all months before creating the final DataFrame
financial_summary_df = {}
# Loop through each loaded file
for filename in SourceFile:

  # Initialize all_actuals_summary_rows as a list for each file to collect its data
  all_actuals_summary_rows = []
  RevisedMonthList[filename] = []
  for current_month in MonthList[filename]:
      # Update BudgetMonth for the current iteration (this variable is used in other functions)
      BudgetMonth = current_month[:3]

      #check if BudgetMonth has actual available, else skip.
      MonthActualsNotFound = False
      for site in SiteList[filename]:
              # Get the DataFrame for the current site
              current_site_df = all_worksheets_df[filename][site]
              # Find LedgerColumn and MonthColumn for the current site using the current BudgetMonth
              if(FindMonthActualsColumn(current_site_df, BudgetMonth)[1] == -1):
                  MonthActualsNotFound = True
                  break
      if(MonthActualsNotFound):
          continue
      else:
        if(current_month not in RevisedMonthList[filename]):
          RevisedMonthList[filename].append(current_month)

      # Iterate through each financial category
      for category in FinCatList:
          # Initialize a dictionary for the current row, including 'Month' and 'Actuals Category'
          row_data = {'File':filename, 'Month': current_month, 'Category': category}

          # Iterate through each site
          for site in SiteList[filename]:
              # Get the DataFrame for the current site
              current_site_df = all_worksheets_df[filename][site]

              # Find LedgerColumn and MonthColumn for the current site using the current BudgetMonth
              current_ledger_col = FindLedgerColumn(current_site_df)
              if ACILFlag[filename] is True:
                current_month_col = MonthColumnsActualsACIL[filename][site][BudgetMonth]
              else:
                current_month_col = FindMonthActualsColumn(current_site_df, BudgetMonth)[1]

              # Calculate the sum for the current site and category
              calculated_sum = calculate_value(
                  DataWorkSheet=current_site_df,
                  LedgerColumnIndex=current_ledger_col,
                  MonthColumnIndex=current_month_col,
                  Category=category
              )
              # Add the calculated sum to the row data with the site name as the key
              row_data[site] = calculated_sum

          # Add the completed row data to the summary list for all months
          all_actuals_summary_rows.append(row_data)

  # Create the final DataFrame from all collected rows for all months
  financial_summary_df[filename] = pd.DataFrame(all_actuals_summary_rows)

  # Display the resulting combined DataFrame
  print("Combined Financial Summary DataFrame (Actuals for all months)-> Filename:",filename)
  print(financial_summary_df)
  print("\n")


Combined Financial Summary DataFrame (Actuals for all months)-> Filename: ACIL-source.xlsx
{'ACIL-source.xlsx':                 File     Month                               Category  \
0   ACIL-source.xlsx  Apr 2026                              Sick Time   
1   ACIL-source.xlsx  Apr 2026                               Overtime   
2   ACIL-source.xlsx  Apr 2026                        Purchased Hours   
3   ACIL-source.xlsx  Apr 2026                           - Added Care   
4   ACIL-source.xlsx  Apr 2026                  Purchased Hours total   
5   ACIL-source.xlsx  Apr 2026                                 Repair   
6   ACIL-source.xlsx  Apr 2026                            Maintenance   
7   ACIL-source.xlsx  Apr 2026                            Other (RnM)   
8   ACIL-source.xlsx  Apr 2026                     6500:Care Supplies   
9   ACIL-source.xlsx  Apr 2026                 6600:Clinical supplies   
10  ACIL-source.xlsx  Apr 2026  Nursing Supplies - Added Care Expense   
11  ACIL-sou

In [209]:
def calculate_budget_sum_general(SiteName, LedgerColumnIndex, TargetCategory, BudgetSheetData):

    # 1. Replicate the MATCH logic on row 15 (Python index 14)
    # Access row 15 (index 14) using .iloc
    header_row = BudgetSheetData.iloc[14]

    target_column_index = -1
    # Enumerate through the values in the header row to find the column index
    for col_idx, cell_value in enumerate(header_row):
        # Truncate each cell value to 3 characters for comparison
        if str(cell_value)[:3].strip().upper() == SiteName:
            target_column_index = col_idx
            break

    # If the column header was not found, return 0.0
    if target_column_index == -1:
        return 0.0

    # 3. Replicate the SUMIF logic on Column A (index 0)
    total_sum = 0.0
    # Iterate through all rows of the DataFrame
    for i in range(BudgetSheetData.shape[0]):
        try:
            # Check if LedgerColumn (index LedgerColumnIndex) matches TargetCategory using .iloc
            # Convert to string and strip whitespace for robust comparison
            if str(BudgetSheetData.iloc[i, LedgerColumnIndex]).strip() == str(TargetCategory).strip():
                # Add the value from the matched target_column_index
                # Convert to numeric, coercing errors to NaN, then add
                total_sum += pd.to_numeric(BudgetSheetData.iloc[i, target_column_index], errors='coerce')
        except (IndexError, ValueError, TypeError):
            # Safe fallback for blanks, headers, or text values in the sum column
            continue

    return total_sum

In [210]:
#given 1)BudgetWorksheet 2)Target Category Column 3)SiteName
#determine which function to use to generate budget value
#then return value from that function



def calculate_budget_value(SiteName, LedgerColumnIndex, TargetCategory, BudgetSheetData):

#Financial Categories (Budget)
#FinCatListBudget = [,"Repair","Maintenance","6500:Care Supplies","6600:Clinical supplies","SUPPLIES","6700:Repair & Maintenance","SUM"]

    StraightBudgetSumCategory = ["Sick Time", "Overtime", "Purchased Hours", "6500:Care Supplies", "6600:Clinical supplies", "6700:Repair & Maintenance"]
    if TargetCategory in StraightBudgetSumCategory:
        return calculate_budget_sum_general(SiteName, LedgerColumnIndex, TargetCategory, BudgetSheetData)

    elif TargetCategory == "Repair" or TargetCategory == "Maintenance":
        return calculate_budget_sum_of_Repair_Maintence(SiteName, LedgerColumnIndex, TargetCategory, BudgetSheetData)

    elif TargetCategory == "SUPPLIES":
        return (calculate_budget_sum_general(SiteName, LedgerColumnIndex, "6500:Care Supplies", BudgetSheetData) + calculate_budget_sum_general(SiteName, LedgerColumnIndex, "6600:Clinical supplies", BudgetSheetData))

    elif TargetCategory == "SUM":
        return (calculate_budget_sum_of_Repair_Maintence(SiteName, LedgerColumnIndex, "Repair", BudgetSheetData) + calculate_budget_sum_of_Repair_Maintence(SiteName, LedgerColumnIndex, "Maintenance", BudgetSheetData))

    else:
        return -1000000

In [211]:
def calculate_budget_sum_of_Repair_Maintence(SiteName, LedgerColumnIndex, TargetCategory, BudgetSheetData):

    # 1. MATCH logic: Find the target data column using the first 3 characters
    match_str = str(SiteName)[:3].strip().upper()
    header_row = BudgetSheetData.iloc[14] # Excel row 15 is index 14

    target_col_idx = -1
    for col_idx, cell_value in enumerate(header_row):
        if str(cell_value)[:3].strip().upper() == match_str:
            target_col_idx = col_idx
            break

    if target_col_idx == -1:
        return 0.0 # Column header not found

    # 2. SCAN + FILTER logic: Loop through rows to find matching section and sub-category
    total_sum = 0.0
    in_target_section = False
    sub_cat_str = str(TargetCategory).strip()
    sub_cat_len = len(sub_cat_str)

    for i in range(BudgetSheetData.shape[0]):
        try:
            # Clean the LedgerColumn value for evaluation
            cell_a_value = str(BudgetSheetData.iloc[i, LedgerColumnIndex]).strip()

            # Replicate the SCAN toggle behavior
            if cell_a_value == "6700:Repair & Maintenance":
                in_target_section = True
            elif in_target_section and ":" in cell_a_value:
                # Any row with a colon (a new header) turns the toggle off
                in_target_section = False

            # Replicate the LEFT(A:A, LEN(B21)) = B21 condition
            # Only match if the string length is sufficient to avoid false matches
            starts_with_sub_cat = cell_a_value[:sub_cat_len] == sub_cat_str if sub_cat_len > 0 else False

            # If both conditions are met, add the column value to our total
            if in_target_section and starts_with_sub_cat:
                total_sum += pd.to_numeric(BudgetSheetData.iloc[i, target_col_idx], errors='coerce')

        except (IndexError, ValueError, TypeError):
            # Gracefully ignore empty rows, text conversion errors, or short rows
            continue

    return total_sum

In [212]:
print(RevisedMonthList)


{'ACIL-source.xlsx': ['Apr 2026']}


In [213]:
def calculate_budget_sum_general_ACIL(LedgerColumnIndex, TargetColumnIndex, TargetCategory, BudgetSheetData):

    # A. Replicate the SUMIF logic on Column A (index 0)
    total_sum = 0.0
    # Iterate through all rows of the DataFrame
    for i in range(BudgetSheetData.shape[0]):
        try:
            # Check if LedgerColumn (index LedgerColumnIndex) matches TargetCategory using .iloc
            # Convert to string and strip whitespace for robust comparison
            if str(BudgetSheetData.iloc[i, LedgerColumnIndex]).strip() == str(TargetCategory).strip():
                # Add the value from the matched TargetColumnIndex
                # Convert to numeric, coercing errors to NaN, then add
                total_sum += pd.to_numeric(BudgetSheetData.iloc[i, TargetColumnIndex], errors='coerce')
        except (IndexError, ValueError, TypeError):
            # Safe fallback for blanks, headers, or text values in the sum column
            continue

    return total_sum

In [214]:
def calculate_budget_sum_of_Repair_Maintence_ACIL(LedgerColumnIndex, TargetColumnIndex, TargetCategory, BudgetSheetData):

    # 2. SCAN + FILTER logic: Loop through rows to find matching section and sub-category
    total_sum = 0.0
    in_target_section = False
    sub_cat_str = str(TargetCategory).strip()
    sub_cat_len = len(sub_cat_str)

    for i in range(BudgetSheetData.shape[0]):
        try:
            # Clean the LedgerColumn value for evaluation
            cell_a_value = str(BudgetSheetData.iloc[i, LedgerColumnIndex]).strip()

            # Replicate the SCAN toggle behavior
            if cell_a_value == "6700:Repair & Maintenance":
                in_target_section = True
            elif in_target_section and ":" in cell_a_value:
                # Any row with a colon (a new header) turns the toggle off
                in_target_section = False

            # Replicate the LEFT(A:A, LEN(B21)) = B21 condition
            # Only match if the string length is sufficient to avoid false matches
            starts_with_sub_cat = cell_a_value[:sub_cat_len] == sub_cat_str if sub_cat_len > 0 else False

            # If both conditions are met, add the column value to our total
            if in_target_section and starts_with_sub_cat:
                total_sum += pd.to_numeric(BudgetSheetData.iloc[i, TargetColumnIndex], errors='coerce')

        except (IndexError, ValueError, TypeError):
            # Gracefully ignore empty rows, text conversion errors, or short rows
            continue

    return total_sum

In [215]:
def calculate_budget_value_ACIL(LedgerColumnIndex, TargetColumnIndex, TargetCategory, BudgetSheetData):
  #Financial Categories (Budget)
  #FinCatListBudget = [,"Repair","Maintenance","6500:Care Supplies","6600:Clinical supplies","SUPPLIES","6700:Repair & Maintenance","SUM"]

    print("X:",LedgerColumnIndex, TargetColumnIndex, TargetCategory, "BudgetSheetData")
    StraightBudgetSumCategory = ["Sick Time", "Overtime", "Purchased Hours", "6500:Care Supplies", "6600:Clinical supplies", "6700:Repair & Maintenance"]
    if TargetCategory in StraightBudgetSumCategory:
        return calculate_budget_sum_general_ACIL(LedgerColumnIndex, TargetColumnIndex, TargetCategory, BudgetSheetData)

    elif TargetCategory == "Repair" or TargetCategory == "Maintenance":
        return calculate_budget_sum_of_Repair_Maintence_ACIL(LedgerColumnIndex, TargetColumnIndex, TargetCategory, BudgetSheetData)

    elif TargetCategory == "SUPPLIES":
        return (calculate_budget_sum_general_ACIL(LedgerColumnIndex, TargetColumnIndex,  "6500:Care Supplies", BudgetSheetData) + calculate_budget_sum_general_ACIL(LedgerColumnIndex, TargetColumnIndex,  "6600:Clinical supplies", BudgetSheetData))

    elif TargetCategory == "SUM":
        return (calculate_budget_sum_of_Repair_Maintence_ACIL(LedgerColumnIndex, TargetColumnIndex,  "Repair", BudgetSheetData) + calculate_budget_sum_of_Repair_Maintence_ACIL(LedgerColumnIndex, TargetColumnIndex,  "Maintenance", BudgetSheetData))

    else:
        return -1000000


In [216]:
# This will store all row data for all months before creating the final DataFrame
financial_summary_budget_df = {}

# Loop through each loaded file
for filename in SourceFile:
  all_budget_summary_rows = []
  # Loop through each month in RevisedMonthList to generate budget data for each
  for current_month in RevisedMonthList[filename]:
      # Update BudgetMonth for the current iteration
      BudgetMonth = current_month[:3]
      if ACILFlag[filename] is False:
        Budget_sheet_name = f"{current_month} Budget"
        worksheet_budget_data = all_worksheets_df[filename][Budget_sheet_name]
        MonthBudgets = BudgetMonth + " Budget"

      # Iterate through each financial category for budget
      for category in FinCatListBudget:
          # Initialize a dictionary for the current row, including 'Month' and 'Budget Category'
          row_data = {'File':filename, 'Month': current_month, 'Category': category}

          # Iterate through each site
          for site in SiteList[filename]:
              # Calculate the sum for the current site and category
              worksheet_data = all_worksheets_df[filename][site]
              if ACILFlag[filename] is True:
                calculated_sum = calculate_budget_value_ACIL(
                    LedgerColumnIndex=FindLedgerColumn(worksheet_data),
                    TargetColumnIndex=MonthColumnsBudgetsACIL[filename][site][BudgetMonth],
                    TargetCategory=category,
                    BudgetSheetData = worksheet_data
                )

              else:
                calculated_sum = calculate_budget_value(
                    SiteName=site,
                    LedgerColumnIndex=FindLedgerColumn(worksheet_data),
                    TargetCategory=category,
                    BudgetSheetData = worksheet_budget_data
                )
              # Add the calculated sum to the row data with the site name as the key
              row_data[site] = calculated_sum

          # Add the completed row data to the summary list for all months
          all_budget_summary_rows.append(row_data)

  # Create the final DataFrame from all collected rows for all months
  financial_summary_budget_df[filename] = pd.DataFrame(all_budget_summary_rows)

  # Display the resulting combined DataFrame
  print("Combined Financial Summary Budget DataFrame (for all months)-> Filename:",filename)
  print(financial_summary_budget_df[filename])


X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Sick Time BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Overtime BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Purchased Hours BudgetSheetData
X: 0 2 Repair BudgetSheetData
X: 0 2 Repair BudgetSheetData
X: 0 2 Repair BudgetSheetData
X: 0 2 Repair BudgetSheetData
X: 0 2 Repair BudgetSheetData
X: 0 2 Repair Budg

In [217]:
# This will store all row data for all months before creating the final DataFrame
financial_summary_variance_df = {}

# Loop through each loaded file
for filename in SourceFile:
  # This will store all row data for all months before creating the final DataFrame
  all_variance_summary_rows = []
  # Loop through each month in RevisedMonthList to generate variance data for each
  for current_month in RevisedMonthList[filename]:
      # Iterate through each variance category
      for category in VarCatList:
          # Initialize a dictionary for the current row, including 'Month' and 'Variance Category'
          row_data = {'File':filename, 'Month': current_month, 'Variance': category}

          # Iterate through each site
          for site in SiteList[filename]:
              # Get the actuals value for the current site and category for the current month
              actuals_filter_category = "Purchased Hours total" if category == "Purchased Hours" else category
              actuals_selection = financial_summary_df[filename].loc[
                  (financial_summary_df[filename]['Month'] == current_month) &
                  (financial_summary_df[filename]['Category'] == actuals_filter_category), site
              ]
              # Safely get actuals_value, defaulting to 0.0 if not found
              actuals_value = actuals_selection.iloc[0] if not actuals_selection.empty else 0.0

              # Get the budget value for the current site and category for the current month
              budget_selection = financial_summary_budget_df[filename].loc[
                  (financial_summary_budget_df[filename]['Month'] == current_month) &
                  (financial_summary_budget_df[filename]['Category'] == category), site
              ]
              # Safely get budget_value, defaulting to 0.0 if not found
              budget_value = budget_selection.iloc[0] if not budget_selection.empty else 0.0

              # Calculate the variance (Budget - Actuals)
              variance = budget_value - actuals_value

              # Add the calculated variance to the row data with the site name as the key
              row_data[site] = variance

          # Add the completed row data to the summary list for all months
          all_variance_summary_rows.append(row_data)

  # Create the final DataFrame from all collected rows for all months
  financial_summary_variance_df[filename] = pd.DataFrame(all_variance_summary_rows)

  # Display the resulting combined DataFrame
  print("Combined Financial Summary Variance DataFrame (for all months)-> Filename:",filename)
  print(financial_summary_variance_df[filename])
  print("\n")



Combined Financial Summary Variance DataFrame (for all months)-> Filename: ACIL-source.xlsx
               File     Month                Variance       CL        GL  \
0  ACIL-source.xlsx  Apr 2026               Sick Time -1542.88  -1587.33   
1  ACIL-source.xlsx  Apr 2026                Overtime -1713.19  -1659.27   
2  ACIL-source.xlsx  Apr 2026         Purchased Hours   208.62    -52.50   
3  ACIL-source.xlsx  Apr 2026                  Repair -5534.49   -114.11   
4  ACIL-source.xlsx  Apr 2026             Maintenance  8529.32  -5132.57   
5  ACIL-source.xlsx  Apr 2026      6500:Care Supplies    59.96  -6409.48   
6  ACIL-source.xlsx  Apr 2026  6600:Clinical supplies   413.61  -6364.29   
7  ACIL-source.xlsx  Apr 2026                SUPPLIES   473.57 -12773.77   

     ACILDS        MN        SN       OM       VV        HC  
0   6068.39 -46080.40  60518.61 -4441.14 -3730.66   -697.80  
1   4663.98 -18771.87 -41088.93 -2330.43  -345.78      9.99  
2      0.00    -52.50      0.00   -52

Now, let's create a new Excel file containing only the data values from the 'Payroll Budget' worksheet. We'll use the DataFrame `df` which already holds these values.

In [218]:


# Helper functions for color gradient
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb_color):
    return '#%02x%02x%02x' % rgb_color

def interpolate_color(start_hex, end_hex, factor):
    start_rgb = hex_to_rgb(start_hex)
    end_rgb = hex_to_rgb(end_hex)
    interpolated_rgb = tuple(
        int(start_rgb[i] + factor * (end_rgb[i] - start_rgb[i])) for i in range(3)
    )
    return rgb_to_hex(interpolated_rgb)

def get_contrasting_font_color(bg_hex_color):
    r, g, b = hex_to_rgb(bg_hex_color)
    # Normalize RGB values to 0-1
    r /= 255.0
    g /= 255.0
    b /= 255.0
    # Convert RGB to HSL and check lightness (L)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    # Use black font for light backgrounds (L > 0.5), white for dark backgrounds
    return '#000000' if l > 0.5 else '#FFFFFF'


In [219]:

# write the Summary File

# Create a Pandas Excel writer using XlsxWriter as the engine.
with pd.ExcelWriter(new_excel_file_path, engine='xlsxwriter') as writer:
    # Get the workbook object to add formats
    workbook = writer.book

    # Combine all months from all files, deduplicate, and sort them
    all_months = []
    for filename in SourceFile:
        all_months.extend(RevisedMonthList[filename])
    unique_sorted_months = sorted(list(set(all_months)), key=lambda x: MonthSequence.index(x))

    # Gradient formats for actuals increases (worse, red shades)
    actuals_increase_formats = [] # This list will hold formats for RED gradient (worse)
    start_color_red = '#FFFFFF' # White
    end_color_red = '#FF0000' # Bright Red
    for i in range(100):
        factor = i / 99.0 # Factor from 0.0 to 1.0
        bg_color = interpolate_color(start_color_red, end_color_red, factor)
        font_color = get_contrasting_font_color(bg_color)
        actuals_increase_formats.append(workbook.add_format({'bg_color': bg_color, 'font_color': font_color, 'num_format': '0.00'}))

    # Gradient formats for actuals decreases (better, green shades)
    actuals_decrease_formats = [] # This list will hold formats for GREEN gradient (better)
    start_color_green = '#FFFFFF' # White
    end_color_green = '#008000' # Dark Green
    for i in range(100):
        factor = i / 99.0
        bg_color = interpolate_color(start_color_green, end_color_green, factor)
        font_color = get_contrasting_font_color(bg_color)
        actuals_decrease_formats.append(workbook.add_format({'bg_color': bg_color, 'font_color': font_color, 'num_format': '0.00'}))

    # Variance formats (simple red/green for now, will be replaced by gradients)
    red_format_variance_negative = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'num_format': '0.00'})
    green_format_variance_positive = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'num_format': '0.00'})

    default_number_format = workbook.add_format({'num_format': '0.00'})
    header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1})
    category_format = workbook.add_format({'bold': True})

    # Cache for previous month's actuals DataFrame to enable comparison (reset for each workbook if needed)
    previous_month_actuals_df_cache = None

    # Now iterate through the unique, sorted list of months to create worksheets
    for idx, Worksheetmonth in enumerate(unique_sorted_months):
        # Create a worksheet for the current month
        worksheet = workbook.add_worksheet(Worksheetmonth)

        # Loop through each loaded file to populate data for the current month
        for filename in SourceFile:

            # Check if Worksheetmonth exists in RevisedMonthList[filename], continue otherwise
            if Worksheetmonth not in RevisedMonthList[filename]:
                continue

            # Filter data for the current month and drop 'File' and 'Month' columns
            # Keep 'Category' for actuals and budget, 'Variance' for variance
            monthly_actuals_df_filtered = financial_summary_df[filename][financial_summary_df[filename]['Month'] == Worksheetmonth].drop(columns=['File', 'Month'])
            monthly_budget_df_filtered = financial_summary_budget_df[filename][financial_summary_budget_df[filename]['Month'] == Worksheetmonth].drop(columns=['File', 'Month'])
            monthly_variance_df_filtered = financial_summary_variance_df[filename][financial_summary_variance_df[filename]['Month'] == Worksheetmonth].drop(columns=['File', 'Month']) # 'Variance' is the category-like column

            # --- Write monthly_actuals_df ---
            startrow_actuals = 0 # Starting row for actuals
            # Write header for actuals
            for col_num, value in enumerate(monthly_actuals_df_filtered.columns.values):
                worksheet.write(startrow_actuals, col_num, value, header_format)

            # Write data rows for actuals with conditional formatting
            for row_num_df, (index, row_data) in enumerate(monthly_actuals_df_filtered.iterrows()):
                excel_row = startrow_actuals + row_num_df + 1 # Excel row, after header
                worksheet.write(excel_row, 0, row_data['Category'], category_format) # Write 'Category' column

                # Compare and write site columns (starting from the second column of the filtered df)
                # The first column is 'Category', so `columns[1:]` gives site names.
                for col_num_df, col_name in enumerate(monthly_actuals_df_filtered.columns[1:]):
                    excel_col = col_num_df + 1 # Excel column, after Category column

                    current_value = row_data[col_name]
                    format_to_apply_actuals = default_number_format # Default format

                    if idx > 0 and previous_month_actuals_df_cache is not None:
                        # Ensure previous_month_actuals_df_cache is also filtered similarly
                        prev_row_selection = previous_month_actuals_df_cache[previous_month_actuals_df_cache['Category'] == row_data['Category']]
                        previous_value = prev_row_selection.iloc[0].get(col_name) if not prev_row_selection.empty else None

                        if previous_value is not None and pd.notna(previous_value) and pd.notna(current_value):
                            if current_value > previous_value: # Actuals INCREASE (worse, red gradient)
                                percent_change = 0.0
                                if previous_value != 0:
                                    percent_change = ((current_value - previous_value) / previous_value) * 100
                                else:
                                    # If previous_value was 0 and current_value is > 0, it's an infinite increase (worse)
                                    percent_change = 200.0 # Map to darkest red for infinite increase from zero

                                # Map 0-200% increase to 0-99 index for the red gradient
                                # Cap percentage at 200% to fit the 100-step gradient nicely.
                                gradient_index = int(min(99, max(0, percent_change / 2)))
                                format_to_apply_actuals = actuals_increase_formats[gradient_index] # This is the RED gradient

                            elif current_value < previous_value: # Actuals DECREASE (better, green gradient)
                                percent_change = 0.0
                                if previous_value != 0:
                                    percent_change = ((previous_value - current_value) / abs(previous_value)) * 100
                                else:
                                    # If previous_value was 0 and current_value is < 0, it's an infinite decrease (better if it's an expense)
                                    percent_change = 100.0 # Map to darkest green for infinite decrease from zero

                                # Map 0-100% decrease to 0-99 index for the green gradient
                                # Cap percentage at 100% to fit the 100-step gradient nicely.
                                gradient_index = int(min(99, max(0, percent_change)))
                                format_to_apply_actuals = actuals_decrease_formats[gradient_index] # This is the GREEN gradient
                            else: # current_value == previous_value
                                format_to_apply_actuals = default_number_format

                    # Write the number with the chosen format, handling NaN values
                    if pd.isna(current_value):
                        worksheet.write_blank(excel_row, excel_col, None, format_to_apply_actuals)
                    else:
                        worksheet.write_number(excel_row, excel_col, current_value, format_to_apply_actuals)

            # Cache current actuals for the next iteration (store the filtered version)
            # This cache should be within the unique_sorted_months loop, but its use seems to imply a single previous month for comparison,
            # which means this logic needs careful consideration if 'previous_month_actuals_df_cache' is meant to be across files.
            # For now, it's reset per unique month, but if cross-file previous month comparison is intended, this needs adjustment.
            # For now, let's assume it should cache per month, for comparison to the 'previous' *unique* month.
            previous_month_actuals_df_cache = monthly_actuals_df_filtered.copy() # This cache will store the last df for the current month across all files. Needs to be more robust.

            # --- Write monthly_budget_df ---
            startrow_budget_monthly = startrow_actuals + len(monthly_actuals_df_filtered) + 2
            # Write header for budget
            for col_num, value in enumerate(monthly_budget_df_filtered.columns.values):
                worksheet.write(startrow_budget_monthly, col_num, value, header_format)
            # Write data rows for budget
            for row_num_df, (index, row_data) in enumerate(monthly_budget_df_filtered.iterrows()):
                excel_row = startrow_budget_monthly + row_num_df + 1
                worksheet.write(excel_row, 0, row_data['Category'], category_format)
                for col_num_df, col_name in enumerate(monthly_budget_df_filtered.columns[1:]):
                    excel_col = col_num_df + 1
                    budget_value = row_data[col_name]
                    if pd.isna(budget_value):
                        worksheet.write_blank(excel_row, excel_col, None, default_number_format)
                    else:
                        worksheet.write_number(excel_row, excel_col, budget_value, default_number_format)


            # --- Write monthly_variance_df --- (Apply gradient formatting)
            startrow_variance_monthly = startrow_budget_monthly + len(monthly_budget_df_filtered) + 2
            # Write header for variance
            for col_num, value in enumerate(monthly_variance_df_filtered.columns.values):
                worksheet.write(startrow_variance_monthly, col_num, value, header_format)

            # Write data rows for variance with gradient conditional formatting
            for row_num_df, (index, row_data) in enumerate(monthly_variance_df_filtered.iterrows()):
                excel_row = startrow_variance_monthly + row_num_df + 1
                category_name = row_data['Variance'] # For variance, this column holds the variance category
                worksheet.write(excel_row, 0, category_name, category_format)

                for col_num_df, col_name in enumerate(monthly_variance_df_filtered.columns[1:]):
                    excel_col = col_num_df + 1
                    variance_value = row_data[col_name]

                    format_to_apply_variance = default_number_format # Default format

                    if pd.notna(variance_value): # Only apply conditional formatting if value is not NaN
                        if variance_value > 0: # Positive variance (better, green gradient)
                            # Map 0 to 30000 to 0-99 index for the green gradient
                            factor = min(1.0, max(0.0, variance_value / 30000.0))
                            gradient_index = int(factor * 99)
                            format_to_apply_variance = actuals_decrease_formats[gradient_index]

                        elif variance_value < 0: # Negative variance (worse, red gradient)
                            # Map -30000 to 0 to 0-99 index for the red gradient
                            factor = min(1.0, max(0.0, abs(variance_value) / 30000.0))
                            gradient_index = int(factor * 99)
                            format_to_apply_variance = actuals_increase_formats[gradient_index]

                    # Write the number with the chosen format, handling NaN values
                    if pd.isna(variance_value):
                        worksheet.write_blank(excel_row, excel_col, None, format_to_apply_variance)
                    else:
                        worksheet.write_number(excel_row, excel_col, variance_value, format_to_apply_variance)

    print(f"Successfully created new Excel file: {DestinationFile} in {DestinationFilePath}")
    print("This file contains Actuals, Budget, and Variance data, with each month on its own sheet.")

(10, 9)
Successfully created new Excel file: Carlos-Summary.xlsx in /content/
This file contains Actuals, Budget, and Variance data, with each month on its own sheet.
